## Sucursales bancarias 2020

En este notebook utilizaremos web scraping para extraer el número de sucursales bancarias que había por municipio en el año 2020.  
Obtendremos un DataFrame con la siguiente información:  
   - Y2020
   - Nombre Provincia
   - Codigo Provincia
   - Nombre municipio
   - Código municipio
   - Nº Sucursales Bancarias

In [1]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.support.ui import Select
import time
import glob

Definimos la página a scrapear y ruta de descarga del chromedriver:

In [2]:
website = 'https://app.bde.es/exbwciu/exbwciuias/xml/Arranque.html#'

In [3]:
path = '/home/dsc/Downloads/chromedriver'

Definimos variable driver y abrimos Google Chrome:

In [4]:
driver = webdriver.Chrome(path)

In [5]:
driver.get(website)

Seleccionamos el informe estadístico y la fecha de los datos a extraer:

In [6]:
OficinasBancarias = driver.find_element_by_xpath('//*[@id="MapaDeLaAplicacioncolMapa_0"]/ul/li[2]/ul/li[2]/a')

In [7]:
OficinasBancarias.click()

In [8]:
Fecha = Select(driver.find_element_by_id('ComboFechaReferencia'))

In [9]:
Fecha.select_by_visible_text('30/12/2020')

Del despegable Provincias, hay que descargar 52 csv, uno por provinica y ciudad autónoma:

In [10]:
for i in range(1, 53):
    Provincias = Select(driver.find_element_by_id('ComboProvincia'))
    Provincias.select_by_index(i)
    Descargar = driver.find_element_by_id('BotonSolicitar')
    Descargar.click()
    Limpiar = driver.find_element_by_id('BotonLimpiar')
    Limpiar.click()
    time.sleep(5)

In [11]:
driver.quit()

Generamos un DataFrame concatenando los csv de cada una de las provincias y ciudades autónomas.  
Descartamos de cada uno de los csv las últimas líneas por contener información no necesaria:

In [12]:
dfLista = []
ruta = '/home/dsc/Downloads/' 
csv_files = glob.glob(ruta + "/*.csv")
count = 0

for file in csv_files:
    with open(file, 'r', encoding='latin1') as f:
        TotalLineas = 0
        for linea in f:
            TotalLineas+= 1
        count += 1
        if count == 1: # primer cvs tiene encabezado
            Inicio = TotalLineas-7
        else: # resto de csv se procesan sin encabezado
            Inicio = TotalLineas-6
           
        df = pd.read_csv(file, encoding='latin1', skiprows = range(Inicio, TotalLineas), sep = ';', 
            usecols = [' Cód. Provincia ', ' Cód. Municipio ', ' Municipio ', ' Nº de oficinas ']) 
        dfLista.append(df)
        
SucursalesBancarias20 = pd.concat(dfLista, ignore_index = True)

In [13]:
SucursalesBancarias20.head()

,Cód. Provincia,Cód. Municipio,Municipio,Nº de oficinas
0,15,30,CORUÑA (A),1
1,15,78,SANTIAGO DE COMPOSTELA,1
2,15,2,AMES,1
3,15,2,AMES,1
4,15,4,ARES,1


Creamos la variable Codigo Municipio uniendo los campos Cód.Provincia y Cód.Municipio.  
El Codigo Municipio se compone de 5 caracteres: 2 Cód.Provincia + 3 Cód.Municipio.  
Añadimos ceros al princpio de Cód.Provincia y Cód.Municipio para generar dicho código.

In [14]:
SucursalesBancarias20.dtypes

 Cód. Provincia      int64
 Cód. Municipio      int64
 Municipio          object
 Nº de oficinas      int64
dtype: object

In [15]:
SucursalesBancarias20[[' Cód. Provincia ', ' Cód. Municipio ' ]] = SucursalesBancarias20[[' Cód. Provincia ', ' Cód. Municipio ' ]].astype(str)

In [16]:
SucursalesBancarias20[' Cód. Provincia '] = SucursalesBancarias20[' Cód. Provincia '].map(lambda x: '0' + x if len(x) == 1 else x)
SucursalesBancarias20[' Cód. Municipio '] = SucursalesBancarias20[' Cód. Municipio '].map(lambda x: '00' + x if len(x) == 1 else ( '0' + x if len(x) == 2 else x))

In [17]:
SucursalesBancarias20['Codigo Municipio'] = SucursalesBancarias20[' Cód. Provincia '] + SucursalesBancarias20[' Cód. Municipio ']

In [18]:
SucursalesBancarias20.head()

,Cód. Provincia,Cód. Municipio,Municipio,Nº de oficinas,Codigo Municipio
0,15,030,CORUÑA (A),1,15030
1,15,078,SANTIAGO DE COMPOSTELA,1,15078
2,15,002,AMES,1,15002
3,15,002,AMES,1,15002
4,15,004,ARES,1,15004


Sumamos el número de oficinas agrupado por municipio:

In [19]:
SucursalesBancarias20 = SucursalesBancarias20.groupby('Codigo Municipio')[' Nº de oficinas '].sum()
SucursalesBancarias20 = SucursalesBancarias20.reset_index()

In [20]:
SucursalesBancarias20.head()

,Codigo Municipio,Nº de oficinas
0,01001,2
1,01002,7
2,01003,2
3,01004,1
4,01009,1


Cargamos la tabla 01_Output_ProvMun_20.xlsx y hacemos un merge con SucursalesBancarias20:

In [21]:
ProvMun = pd.read_excel('01_Output_ProvMun_20.xlsx', dtype='object') 
ProvMun.head()

,Nombre Provincia,Codigo Provincia,Nombre Municipio,Codigo Municipio
0,Araba/Álava,01,Agurain/Salvatierra,01051
1,Araba/Álava,01,Alegría-Dulantzi,01001
2,Araba/Álava,01,Amurrio,01002
3,Araba/Álava,01,Añana,01049
4,Araba/Álava,01,Aramaio,01003


In [22]:
SucursalesBancarias20 = ProvMun.merge(SucursalesBancarias20, on='Codigo Municipio')
SucursalesBancarias20.head()

,Nombre Provincia,Codigo Provincia,Nombre Municipio,Codigo Municipio,Nº de oficinas
0,Araba/Álava,01,Agurain/Salvatierra,01051,4
1,Araba/Álava,01,Alegría-Dulantzi,01001,2
2,Araba/Álava,01,Amurrio,01002,7
3,Araba/Álava,01,Aramaio,01003,2
4,Araba/Álava,01,Arraia-Maeztu,01037,1


Cambiamos el nombre de la columna Nº de oficinas y añadirmos una nueva columna con el año 2020:

In [23]:
SucursalesBancarias20.rename({' Nº de oficinas ': 'Nº Sucursales Bancarias'}, axis=1, inplace = True)
SucursalesBancarias20['Y2020'] = 2020
PrimeraColumna = SucursalesBancarias20.pop('Y2020')
SucursalesBancarias20.insert(0,'Y2020', PrimeraColumna)
SucursalesBancarias20.head()

,Y2020,Nombre Provincia,Codigo Provincia,Nombre Municipio,Codigo Municipio,Nº Sucursales Bancarias
0,2020,Araba/Álava,01,Agurain/Salvatierra,01051,4
1,2020,Araba/Álava,01,Alegría-Dulantzi,01001,2
2,2020,Araba/Álava,01,Amurrio,01002,7
3,2020,Araba/Álava,01,Aramaio,01003,2
4,2020,Araba/Álava,01,Arraia-Maeztu,01037,1


In [24]:
SucursalesBancarias20.to_excel('17_Output_Sucursales_Bancarias_20.xlsx', header = True, index = False)